In [1]:
import sys
import time
import platform
import os
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

print("=" * 60)
print("COMPUTATIONAL ENVIRONMENT")
print("=" * 60)

print("Python version :", sys.version)
print("Platform       :", platform.platform())
print("Processor      :", platform.processor())
print("CPU cores      :", os.cpu_count())
print("NumPy          :", np.__version__)
print("Pandas         :", pd.__version__)
print("Scikit-learn  :", sklearn.__version__)
print("XGBoost        :", xgb.__version__)

COMPUTATIONAL ENVIRONMENT
Python version : 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
Platform       : Windows-10-10.0.26200-SP0
Processor      : Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
CPU cores      : 12
NumPy          : 2.2.6
Pandas         : 2.3.3
Scikit-learn  : 1.7.2
XGBoost        : 3.2.0


In [2]:
import pandas as pd
import numpy as np

save_path = r"C:\Users\Asus\OneDrive\Desktop\neopain-research\data\processed"

df = pd.read_csv(
    f"{save_path}\\ucl_feature_matrix_clean.csv"
)

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

meta_cols = [
    'mat_path',
    'EEG file name',
    'Record identifier',
    'condition',
    'pain_label',
    'pipp_label',
    'Total PIPP score',
    'age_group',
    'on_analgesic'
]

feat_cols = [
    c for c in df.columns
    if c not in meta_cols
]

X = df[feat_cols].values
y = df['pain_label'].values
subjects = df['Record identifier'].values

print("\nComputational dataset")
print("--------------------")
print("Samples        :", X.shape[0])
print("Features       :", X.shape[1])
print("Unique subjects:", len(np.unique(subjects)))

Dataset shape: (295, 78)
Columns: 78

Computational dataset
--------------------
Samples        : 295
Features       : 69
Unique subjects: 105


In [3]:
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=(y == 0).sum() / max((y == 1).sum(), 1),
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)

print("=" * 60)
print("XGBOOST CONFIGURATION")
print("=" * 60)

print("n_estimators     :", model.get_params()["n_estimators"])
print("max_depth        :", model.get_params()["max_depth"])
print("learning_rate    :", model.get_params()["learning_rate"])
print("eval_metric      :", model.get_params()["eval_metric"])
print("random_state     :", model.get_params()["random_state"])
print("scale_pos_weight :", model.get_params()["scale_pos_weight"])

XGBOOST CONFIGURATION
n_estimators     : 200
max_depth        : 4
learning_rate    : 0.05
eval_metric      : logloss
random_state     : 42
scale_pos_weight : 1.8640776699029127


In [4]:
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import numpy as np
import time

unique_subj = np.unique(subjects)

training_times = []
inference_times = []

all_auc = []
all_f1 = []

for subj in unique_subj:

    test_mask = subjects == subj
    train_mask = ~test_mask

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    # Skip subjects with only one class
    if len(np.unique(y_test)) < 2:
        continue

    # --------------------------------------------------
    # Scaling — same procedure as original LOSO pipeline
    # --------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------
    # XGBoost — same configuration as original model
    # --------------------------------------------------

    scale_pw = (
        (y_train == 0).sum()
        / max((y_train == 1).sum(), 1)
    )

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pw,
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    )

    # --------------------------------------------------
    # Training time
    # --------------------------------------------------

    start_train = time.perf_counter()

    model.fit(
        X_train_scaled,
        y_train
    )

    end_train = time.perf_counter()

    train_time = end_train - start_train
    training_times.append(train_time)

    # --------------------------------------------------
    # Inference time
    # --------------------------------------------------

    start_inference = time.perf_counter()

    y_prob = model.predict_proba(
        X_test_scaled
    )[:, 1]

    y_pred = model.predict(
        X_test_scaled
    )

    end_inference = time.perf_counter()

    inference_time = end_inference - start_inference
    inference_times.append(inference_time)

    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------

    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    all_auc.append(auc)
    all_f1.append(f1)

# ------------------------------------------------------
# Summary
# ------------------------------------------------------

print("=" * 60)
print("COMPUTATIONAL ANALYSIS — LOSO")
print("=" * 60)

print("Evaluated subjects :", len(training_times))

print("\nTraining time")
print("-------------")
print("Total   :", f"{np.sum(training_times):.3f} seconds")
print("Mean    :", f"{np.mean(training_times):.3f} seconds")
print("Median  :", f"{np.median(training_times):.3f} seconds")
print("Min     :", f"{np.min(training_times):.3f} seconds")
print("Max     :", f"{np.max(training_times):.3f} seconds")

print("\nInference time")
print("--------------")
print("Total   :", f"{np.sum(inference_times):.3f} seconds")
print("Mean    :", f"{np.mean(inference_times):.5f} seconds")
print("Median  :", f"{np.median(inference_times):.5f} seconds")
print("Min     :", f"{np.min(inference_times):.5f} seconds")
print("Max     :", f"{np.max(inference_times):.5f} seconds")

COMPUTATIONAL ANALYSIS — LOSO
Evaluated subjects : 95

Training time
-------------
Total   : 22.325 seconds
Mean    : 0.235 seconds
Median  : 0.213 seconds
Min     : 0.176 seconds
Max     : 0.522 seconds

Inference time
--------------
Total   : 0.214 seconds
Mean    : 0.00225 seconds
Median  : 0.00204 seconds
Min     : 0.00172 seconds
Max     : 0.00573 seconds


In [5]:
# Train one final model on the complete dataset
# only for computational/model-size characterization.

scaler_final = StandardScaler()

X_scaled = scaler_final.fit_transform(X)

scale_pw_final = (
    (y == 0).sum()
    / max((y == 1).sum(), 1)
)

final_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pw_final,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)

start = time.perf_counter()

final_model.fit(
    X_scaled,
    y
)

fit_time_final = time.perf_counter() - start

print("=" * 60)
print("FINAL MODEL CHARACTERISTICS")
print("=" * 60)

print("Number of features :", X.shape[1])
print("Number of trees    :", final_model.get_booster().num_boosted_rounds())
print("Max tree depth     :", final_model.max_depth)
print("Training time      :", f"{fit_time_final:.3f} seconds")

FINAL MODEL CHARACTERISTICS
Number of features : 69
Number of trees    : 200
Max tree depth     : 4
Training time      : 0.238 seconds


In [6]:
computational_summary = pd.DataFrame({
    "parameter": [
        "samples",
        "features",
        "unique_subjects",
        "xgboost_estimators",
        "xgboost_max_depth",
        "xgboost_learning_rate",
        "mean_loso_training_time_sec",
        "median_loso_training_time_sec",
        "mean_loso_inference_time_sec",
        "median_loso_inference_time_sec",
        "final_model_training_time_sec"
    ],
    "value": [
        X.shape[0],
        X.shape[1],
        len(np.unique(subjects)),
        200,
        4,
        0.05,
        np.mean(training_times),
        np.median(training_times),
        np.mean(inference_times),
        np.median(inference_times),
        fit_time_final
    ]
})

print(computational_summary)

computational_summary.to_csv(
    f"{save_path}\\computational_analysis.csv",
    index=False
)

print("\nSaved:")
print(f"{save_path}\\computational_analysis.csv")

                         parameter       value
0                          samples  295.000000
1                         features   69.000000
2                  unique_subjects  105.000000
3               xgboost_estimators  200.000000
4                xgboost_max_depth    4.000000
5            xgboost_learning_rate    0.050000
6      mean_loso_training_time_sec    0.235003
7    median_loso_training_time_sec    0.213308
8     mean_loso_inference_time_sec    0.002254
9   median_loso_inference_time_sec    0.002038
10   final_model_training_time_sec    0.238376

Saved:
C:\Users\Asus\OneDrive\Desktop\neopain-research\data\processed\computational_analysis.csv
